# Experiment [6] Visual Dashboard: Strict Hybrid Raw-Cue Probe

Probe-only strict heldout evaluation over the repaired `[4]` JEPA checkpoints. The point is to see whether adding compact raw spectral/SAR summary statistics at probe time closes the gap to stronger external baselines or mainly exposes a localized LEM Brazil issue.

This notebook reads the pulled artifacts under `artifacts/[6]/strict_hybrid_raw_cue_probe/`. It does not train anything.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN = ROOT / "artifacts" / "[6]" / "strict_hybrid_raw_cue_probe"
EXTERNAL = ROOT / "artifacts" / "[5]" / "repaired_external_v3"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "surf_jepa_v0_plus_raw_stats": "#059669",
    "raw_stats": "#f59e0b",
    "raw_flattened": "#64748b",
    "olmoearth": "#7c3aed",
    "presto": "#db2777",
}

METRIC_LABELS = {
    "mean_f1": "F1",
    "mean_auc": "AUROC",
    "mean_balanced_accuracy": "Balanced accuracy",
    "mean_calibrated_f1": "Calibrated F1",
    "mean_calibrated_balanced_accuracy": "Calibrated balanced accuracy",
}


def read_csv(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def scorecard(df, title, sort_by=None):
    if df.empty:
        print(f"{title}: no rows")
        return df
    show = df.copy()
    if sort_by and sort_by in show.columns:
        show = show.sort_values(sort_by, ascending=False)
    display_cols = [c for c in [
        "config", "arm", "holdout", "condition", "label_budget",
        "mean_f1", "mean_auc", "mean_balanced_accuracy",
        "mean_calibrated_f1", "mean_calibrated_balanced_accuracy",
        "n_rows", "n_seeds", "n_holdouts",
    ] if c in show.columns]
    print(title)
    display(show[display_cols])
    return show


def bar_metric(df, x, y, color, title, barmode="group", height=480):
    if df.empty:
        return None
    fig = px.bar(df, x=x, y=y, color=color, barmode=barmode, text_auto=".3f", color_discrete_map=PALETTE)
    fig.update_layout(title=title, height=height, xaxis_title="", yaxis_title=METRIC_LABELS.get(y, y), legend_title_text="")
    fig.show()
    return fig


def line_metric(df, x, y, color, title, facet_col=None, height=520):
    if df.empty:
        return None
    fig = px.line(df, x=x, y=y, color=color, facet_col=facet_col, markers=True, color_discrete_map=PALETTE)
    fig.update_layout(title=title, height=height, yaxis_title=METRIC_LABELS.get(y, y), legend_title_text="")
    fig.show()
    return fig


def heatmap_metric(df, index, columns, values, title, height=520):
    if df.empty:
        return None
    pivot = df.pivot_table(index=index, columns=columns, values=values, aggfunc="mean")
    fig = px.imshow(pivot, text_auto=".3f", aspect="auto", color_continuous_scale="Viridis")
    fig.update_layout(title=title, height=height, xaxis_title="", yaxis_title="")
    fig.show()
    display(pivot)
    return fig


## 1. Load Artifacts

First verify that the pulled run has the expected checkpoint manifest and row count.

In [ ]:
metadata = read_json(RUN / "metadata.json")
manifest = read_csv(RUN / "checkpoint_manifest.csv")
results = read_csv(RUN / "probe_results.csv")
summary = read_csv(RUN / "probe_summary.csv")
priority = read_csv(RUN / "priority_summary.csv")
per_holdout = read_csv(RUN / "per_holdout_priority_summary.csv")
lem_priority = read_csv(RUN / "lem_brazil_priority_summary.csv")
curves = read_csv(RUN / "label_budget_curves.csv")

print(json.dumps(metadata, indent=2))
print(f"manifest rows: {len(manifest)}")
print(f"probe rows: {len(results)}")
print(f"summary rows: {len(summary)}")
scorecard(manifest, "Checkpoint manifest")


## 2. Priority Scorecard

The headline question: does `surf_jepa_v0_plus_raw_stats` clear the planned priority threshold and improve AUROC over embedding-only JEPA?

In [ ]:
priority_view = scorecard(priority, "[6] Priority aggregate", sort_by="mean_f1")

long = priority.melt(
    id_vars=["config", "arm"],
    value_vars=["mean_f1", "mean_auc", "mean_balanced_accuracy", "mean_calibrated_f1"],
    var_name="metric",
    value_name="value",
)
fig = px.bar(
    long,
    x="metric",
    y="value",
    color="arm",
    facet_col="config",
    barmode="group",
    text_auto=".3f",
    color_discrete_map=PALETTE,
)
fig.update_layout(title="[6] Priority metrics by arm", height=520, xaxis_title="", yaxis_title="value", legend_title_text="")
fig.show()


## 3. Hybrid Delta

Compute the direct hybrid-minus-embedding delta for each config. This is the clearest read on whether raw cue exposure helps.

In [ ]:
def arm_delta(df, base="surf_jepa_v0", hybrid="surf_jepa_v0_plus_raw_stats", group_cols=("config",)):
    base_df = df[df.arm == base].copy()
    hybrid_df = df[df.arm == hybrid].copy()
    keys = list(group_cols)
    merged = hybrid_df.merge(base_df, on=keys, suffixes=("_hybrid", "_embedding"))
    out = merged[keys].copy()
    for metric in ["mean_f1", "mean_auc", "mean_balanced_accuracy", "mean_calibrated_f1", "mean_calibrated_balanced_accuracy"]:
        out[f"delta_{metric}"] = merged[f"{metric}_hybrid"] - merged[f"{metric}_embedding"]
        out[f"{metric}_hybrid"] = merged[f"{metric}_hybrid"]
        out[f"{metric}_embedding"] = merged[f"{metric}_embedding"]
    return out

priority_delta = arm_delta(priority, group_cols=("config",))
scorecard(priority_delta.rename(columns={"delta_mean_f1":"mean_f1", "delta_mean_auc":"mean_auc", "delta_mean_calibrated_f1":"mean_calibrated_f1"}), "Hybrid minus embedding delta")

plot_delta = priority_delta.melt(
    id_vars=["config"],
    value_vars=["delta_mean_f1", "delta_mean_auc", "delta_mean_balanced_accuracy", "delta_mean_calibrated_f1"],
    var_name="metric",
    value_name="delta",
)
fig = px.bar(plot_delta, x="metric", y="delta", color="config", barmode="group", text_auto=".3f")
fig.add_hline(y=0, line_width=1, line_color="#334155")
fig.update_layout(title="[6] Hybrid minus embedding delta", height=460, xaxis_title="", yaxis_title="delta")
fig.show()


## 4. Stress Conditions

If raw stats help only on clean data, that is less useful. Here we check whether hybrid gains survive the sensor/time stress conditions.

In [ ]:
condition_delta = arm_delta(summary, group_cols=("config", "holdout", "condition"))
condition_agg = condition_delta.groupby(["config", "condition"], as_index=False).agg(
    delta_f1=("delta_mean_f1", "mean"),
    delta_auc=("delta_mean_auc", "mean"),
    hybrid_f1=("mean_f1_hybrid", "mean"),
    embedding_f1=("mean_f1_embedding", "mean"),
    hybrid_auc=("mean_auc_hybrid", "mean"),
    embedding_auc=("mean_auc_embedding", "mean"),
)
display(condition_agg)

fig = px.bar(condition_agg, x="condition", y="delta_f1", color="config", barmode="group", text_auto=".3f")
fig.add_hline(y=0, line_width=1, line_color="#334155")
fig.update_layout(title="[6] Hybrid F1 gain by stress condition", height=460, xaxis_title="", yaxis_title="F1 delta")
fig.show()

fig = px.bar(condition_agg, x="condition", y="delta_auc", color="config", barmode="group", text_auto=".3f")
fig.add_hline(y=0, line_width=1, line_color="#334155")
fig.update_layout(title="[6] Hybrid AUROC gain by stress condition", height=460, xaxis_title="", yaxis_title="AUROC delta")
fig.show()


## 5. Per-Heldout View

This shows where hybrid helps and where it does not. LEM Brazil is the key failure case to inspect.

In [ ]:
scorecard(per_holdout, "[6] Per-holdout priority", sort_by="mean_f1")
heatmap_metric(per_holdout[per_holdout.config == "large_dual_s2"], index="arm", columns="holdout", values="mean_f1", title="[6] large_dual_s2 priority F1 by heldout")
heatmap_metric(per_holdout[per_holdout.config == "large_dual_s2"], index="arm", columns="holdout", values="mean_auc", title="[6] large_dual_s2 priority AUROC by heldout")


## 6. LEM Brazil

LEM Brazil tests whether raw optical/red-edge cues are still not being preserved. The hybrid should help if compact raw stats are sufficient; raw flattened winning means richer temporal/magnitude structure still matters.

In [ ]:
scorecard(lem_priority, "[6] LEM Brazil priority", sort_by="mean_f1")

lem_long = lem_priority.melt(
    id_vars=["config", "arm"],
    value_vars=["mean_f1", "mean_auc", "mean_balanced_accuracy", "mean_calibrated_f1"],
    var_name="metric",
    value_name="value",
)
fig = px.bar(
    lem_long,
    x="metric",
    y="value",
    color="arm",
    facet_col="config",
    barmode="group",
    text_auto=".3f",
    color_discrete_map=PALETTE,
)
fig.update_layout(title="[6] LEM Brazil metrics by arm", height=520, xaxis_title="", yaxis_title="value", legend_title_text="")
fig.show()

lem_condition = summary[(summary.holdout == "lem-brazil") & (summary.config == "large_dual_s2")]
heatmap_metric(lem_condition, index="arm", columns="condition", values="mean_f1", title="[6] LEM Brazil F1 by condition: large_dual_s2")
heatmap_metric(lem_condition, index="arm", columns="condition", values="mean_auc", title="[6] LEM Brazil AUROC by condition: large_dual_s2")


## 7. Label-Budget Curves

The paper story needs sparse-label behavior. These curves average across seeds and heldouts for each condition and arm.

In [ ]:
curve_main = curves[curves.config == "large_dual_s2"].copy()
curve_main["label_budget_pct"] = curve_main["label_budget"] * 100
line_metric(
    curve_main,
    x="label_budget_pct",
    y="mean_f1",
    color="arm",
    facet_col="condition",
    title="[6] large_dual_s2 label-budget F1 curves by condition",
    height=760,
)
line_metric(
    curve_main,
    x="label_budget_pct",
    y="mean_auc",
    color="arm",
    facet_col="condition",
    title="[6] large_dual_s2 label-budget AUROC curves by condition",
    height=760,
)


## 8. External Baseline Context

Optional comparison against fixed `[5]` external baselines. Use this only for context: `[6]` is a SURF-side probe ablation, while `[5]` contains Presto and OlmoEarth frozen external baselines.

In [ ]:
external = read_csv(EXTERNAL / "combined_probe_summary.csv")
if not external.empty:
    ext_priority = external.groupby("baseline", as_index=False).agg(
        mean_f1=("mean_f1", "mean"),
        mean_auc=("mean_auc", "mean"),
        mean_balanced_accuracy=("mean_balanced_accuracy", "mean"),
        mean_calibrated_f1=("mean_calibrated_f1", "mean"),
        mean_calibrated_balanced_accuracy=("mean_calibrated_balanced_accuracy", "mean"),
    )
    surf_context = priority[(priority.config == "large_dual_s2") & (priority.arm.isin(["surf_jepa_v0", "surf_jepa_v0_plus_raw_stats", "raw_stats", "raw_flattened"]))].copy()
    surf_context = surf_context.rename(columns={"arm":"baseline"})
    surf_context["baseline"] = surf_context["baseline"].replace({
        "surf_jepa_v0": "JEPA embedding",
        "surf_jepa_v0_plus_raw_stats": "JEPA + raw stats",
        "raw_stats": "raw stats",
        "raw_flattened": "raw flattened",
    })
    context = pd.concat([surf_context[ext_priority.columns], ext_priority], ignore_index=True)
    scorecard(context, "[6] Context versus [5] external baselines", sort_by="mean_f1")
    long = context.melt(id_vars="baseline", value_vars=["mean_f1", "mean_auc", "mean_calibrated_f1"], var_name="metric", value_name="value")
    fig = px.bar(long, x="metric", y="value", color="baseline", barmode="group", text_auto=".3f", color_discrete_map=PALETTE)
    fig.update_layout(title="[6] Hybrid probe in external-baseline context", height=500, xaxis_title="", yaxis_title="value", legend_title_text="")
    fig.show()
else:
    print("External baseline table not found; skipping context plot.")


## 9. Takeaway

Use this cell as a compact written read after rerunning the notebook.

In [ ]:
best = priority.sort_values("mean_f1", ascending=False).iloc[0]
ld = priority[(priority.config == "large_dual_s2") & (priority.arm == "surf_jepa_v0")].iloc[0]
hy = priority[(priority.config == "large_dual_s2") & (priority.arm == "surf_jepa_v0_plus_raw_stats")].iloc[0]
lem_embed = lem_priority[(lem_priority.config == "large_dual_s2") & (lem_priority.arm == "surf_jepa_v0")].iloc[0]
lem_hybrid = lem_priority[(lem_priority.config == "large_dual_s2") & (lem_priority.arm == "surf_jepa_v0_plus_raw_stats")].iloc[0]
lem_raw = lem_priority[(lem_priority.config == "large_dual_s2") & (lem_priority.arm == "raw_flattened")].iloc[0]

print(f"Best priority arm: {best.config} / {best.arm} F1={best.mean_f1:.4f}, AUROC={best.mean_auc:.4f}")
print(f"large_dual_s2 hybrid gain: F1 {hy.mean_f1 - ld.mean_f1:+.4f}, AUROC {hy.mean_auc - ld.mean_auc:+.4f}")
print(f"LEM hybrid gain: F1 {lem_hybrid.mean_f1 - lem_embed.mean_f1:+.4f}, AUROC {lem_hybrid.mean_auc - lem_embed.mean_auc:+.4f}")
print(f"LEM raw_flattened remains: F1={lem_raw.mean_f1:.4f}, AUROC={lem_raw.mean_auc:.4f}")
